This demonstration presents a complete pipeline for augmenting the [AI4I2020](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) predictive maintenance dataset using a variety of SOTA data generators ([CTGAN](https://github.com/sdv-dev/CTGAN), [TVAE](https://github.com/sdv-dev/CTGAN/blob/main/ctgan/synthesizers/tvae.py), [Gaussian Copula](https://docs.sdv.dev/sdv/single-table-data/modeling/synthesizers/gaussiancopulasynthesizer) and [SMOTENC](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html)) and comparing them to **K-IPO (Kendall-constrained Importance Preserving Oversampling for Imbalanced Tabular Data)**. 

Five different augmented datasets are created, one for each generator. For each dataset, three different estimators are trained:

- [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

- [Multi-Layer Perceptron (MLP)](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

- [GAMI-Net](https://selfexplainml.github.io/PiML-Toolbox/_build/html/guides/models/gaminet.html)

The performance of these models is evaluated both in terms of their **predictive capabilities** and their **explainability**. 

Explainability is assessed based on the resulting **feature importance rankings**: for the Random Forest classifier, feature importance is extracted using [SHAP](https://shap.readthedocs.io/en/latest/api.html); for the MLP model, [SOFI](https://github.com/igraugar/sofi) is used; and for the GAMI-Net classifier, the ranking is obtained directly from the model due to its **inherently interpretable** nature.

After training and evaluating the models on all augmented datasets, the results for each generator are presented.


In [1]:
import logging; import warnings
import os; import yaml; import numpy as np; import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.base import clone

import matplotlib
matplotlib.use("Agg") # To avoid PiML plotting 

from IPython.display import display

from kipo.selector import KIPOSelector as KIPO
from kipo.importance import compute_importance, kendall_tau
from kipo.generator import generate_data_pool
from kipo.models import models
from kipo.utils import load_data, preprocessing, encode_target, extract_target_var, construct_metadata

from experiments.utils import (compute_explanations, compute_final_results, calculate_threshold, compute_metrics, 
                               separability_N3, format_feature_importance, compute_importance, compute_importance_overlap,
                               get_num_topk_features, compute_topk_ordering, validate_ai4i2020)

In [2]:
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("smote_variants").setLevel(logging.WARNING) # To avoid SMOTE warnings regarding deprecation of certain functions due to 3.10
warnings.filterwarnings("ignore")

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

The `config.yml` file contains all configuration settings for the current demonstration. Among other parameters, this file specifies the dataset to be augmented (**AI4I2020**), the desired balance ratio between the two classes of the target variable after augmentation (**0.75**) and the train-test split ratio (**0.8**) to be used.

Additionally, through this file, the values of key parameters controlling the augmentation process with *K-IPO* are specified. These parameters include the desired **Kendall tau threshold**, the minimum number of **top features whose importance ranking should remain unchanged** after augmentation, and the minimum number of **top features for which importance rankings should overlap**.

Finally, the `config.yml` also includes the configuration of parameters for all generators that will be used for dataset augmentation.

In [3]:
with open(os.path.join(os.getcwd(), 'config.yml')) as f:
    demo_config = yaml.safe_load(f)

DATASET = demo_config["dataset"]; BALANCE_RATIO = demo_config["balance_ratio"]; TRAIN_TEST_RATIO = demo_config["train_test_ratio"]; SEED_GEN = demo_config["seed"]
TAU_THRESHOLD = demo_config["tau_threshold"]; TOPK_ORDERING = demo_config["topk_ordering"]; TOPK_OVERLAP = demo_config["topk_overlap"]

GENERATORS_PARAMS = demo_config["gen_params"]

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

Each dataset is described by a dedicated **YAML** file. Among other things, this file contains information about the types of input features (numerical, categorical), specifies the target variable and indicates any columns that should be dropped.

Based on the information provided in the corresponding dataset’s YAML file, appropriate preprocessing (i.e. normalization, encoding) is applied to the input features. 

In [4]:
with open(os.path.join(os.getcwd(), "datasets", "config", f"{os.path.splitext(DATASET)[0]}.yml")) as f:
    dataset_conf = yaml.safe_load(f)

cat_cols = [list(col.keys())[0] for col in dataset_conf.get("cat_cols", [])]; num_cols = dataset_conf.get("num_cols", [])

data = load_data(os.path.join(os.getcwd(), "datasets", "data", DATASET), dataset_conf)
X, y = extract_target_var(data, dataset_conf)

X, pipeline = preprocessing(X, dataset_conf)
y = encode_target(y, dataset_conf)

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

In [5]:
majority_count = int((y == 0).sum().iloc[0])
minority_count = int((y == 1).sum().iloc[0])

total_count = majority_count + minority_count

imbalance_ratio = minority_count / majority_count
majority_pct = majority_count / total_count * 100
minority_pct = minority_count / total_count * 100

num_input_features = X.shape[1]

print(f"[INFO] Dataset: {DATASET}, Number of samples: {X.shape[0]}, Number of input features: {X.shape[1]}")
print(f"[INFO] Numerical input features: {num_cols}")
print(f"[INFO] Categorical input features: {cat_cols}")
print(f"[INFO] Majority class samples: {majority_count} ({majority_pct:.2f}%), Minority class samples: {minority_count} ({minority_pct:.2f}%)")
print(f"[INFO] Imbalance ratio (minority/majority): {imbalance_ratio:.4f}, Desired balance ratio: {BALANCE_RATIO}")

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

[INFO] Dataset: ai4i2020, Number of samples: 9981, Number of input features: 6
[INFO] Numerical input features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
[INFO] Categorical input features: ['Type']
[INFO] Majority class samples: 9643 (96.61%), Minority class samples: 338 (3.39%)
[INFO] Imbalance ratio (minority/majority): 0.0351, Desired balance ratio: 0.75


The K-IPO augmentation process relies on the feature importance ranking of the raw data, as determined by the **ANOVA F-score**. By configuring parameters such as the **Kendall's tau coefficient**, **top-k ordering** and **top-k overlap**, K-IPO ensures that this ranking is preserved at the desired level in the augmented dataset.

In [6]:
raw_importance = compute_importance("f-score_ANOVA", X, y.values.ravel())

print(f"[INFO] Feature importance ranking of raw data based on ANOVA F-score:")
print(raw_importance)

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

[INFO] Feature importance ranking of raw data based on ANOVA F-score:
Torque [Nm]                380.274466
Tool wear [min]            111.144315
Air temperature [K]         68.249136
Rotational speed [rpm]      19.325488
Type                        13.234950
Process temperature [K]     12.868174
dtype: float64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = TRAIN_TEST_RATIO, random_state = SEED_GEN, stratify = y)
    
X_train = X_train.reset_index(drop = True); X_test = X_test.reset_index(drop = True)
y_train = y_train.reset_index(drop = True); y_test = y_test.reset_index(drop = True)

print(f"After splitting with train/test ratio: {TRAIN_TEST_RATIO} -> Training set samples: {X_train.shape[0]}, Test set samples: {X_test.shape[0]}")

majority_class_train_samples = int((y_train == 0).sum().iloc[0]); minority_class_train_samples = int((y_train == 1).sum().iloc[0])
total_needed_samples = max(0, int(majority_class_train_samples * BALANCE_RATIO) - minority_class_train_samples)

print(f"Total number of samples to be generated (based on balance ratio {BALANCE_RATIO}): {total_needed_samples}")

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

After splitting with train/test ratio: 0.8 -> Training set samples: 7984, Test set samples: 1997
Total number of samples to be generated (based on balance ratio 0.75): 5515


In [8]:
ctgan_config = {
    "method": "ctgan",
    "params": {
        **GENERATORS_PARAMS.get("CTGAN", {}),
        **({"discrete_features": cat_cols})
    }
}

tvae_config = {
    "method": "tvae",
    "params": {
        **GENERATORS_PARAMS.get("TVAE", {}),
        **({"discrete_features": cat_cols})
    }
}

gaussian_copula_config = {
    "method": "gaussiancopula",
    "params": {
        **GENERATORS_PARAMS.get("GaussianCopula", {}),
        **({"metadata": construct_metadata(dataset_conf)})
    }
}

smotenc_config = {
    "method": "smotenc",
    "params": {
        **GENERATORS_PARAMS.get("SMOTENC", {}),
        **({"categorical_features": cat_cols})  
    }
}

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

An augmented dataset is obtained for each of the four SOTA generators.

In [ ]:
X_aug_gaussian_copula, y_aug_gaussian_copula = generate_data_pool("gaussiancopula", X_train, y_train, total_needed_samples, kipo = False, **gaussian_copula_config["params"])
X_aug_ctgan, y_aug_ctgan = generate_data_pool("tvae", X_train, y_train, total_needed_samples, kipo = False, **tvae_config["params"])
X_aug_tvae, y_aug_tvae = generate_data_pool("tvae", X_train, y_train, total_needed_samples, kipo = False, **tvae_config["params"])
X_aug_smotenc, y_aug_smotenc = generate_data_pool("smotenc", X_train, y_train, total_needed_samples, kipo = False, **smotenc_config["params"])

augmented_datasets = {"gaussian_copula": (X_aug_gaussian_copula, y_aug_gaussian_copula), "ctgan": (X_aug_ctgan, y_aug_ctgan),
                      "tvae": (X_aug_tvae, y_aug_tvae), "smotenc": (X_aug_smotenc, y_aug_smotenc)}

for method, dataset in augmented_datasets.items():
    try:
        validate_ai4i2020(dataset)

    except Exception as e:
        print(f"[{method}] Augmented dataset validation failed: {e}"); raise

print(f"Generation of augmented data without K-IPO completed [generators: CTGAN, TVAE, GaussianCopula, SMOTENC]")

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

Generation of augmented data without K-IPO completed [generators: CTGAN, TVAE, GaussianCopula, SMOTENC]


Augmentation with K-IPO is performed using a **0.95** Kendall tau threshold, ensuring that the ANOVA F-value-based feature importance ranking remains unchanged for at least the top **two** features and that there is overlap among at least the top **four** features.

In [22]:
print(f"Generation of augmented data with K-IPO")
print(f"Kendall's Tau threshold: {TAU_THRESHOLD}, Topk ordering: {TOPK_ORDERING}, Topk overlap: {TOPK_OVERLAP}")

kipo = KIPO(num_input_features, tau_threshold = TAU_THRESHOLD, topk_ordering = TOPK_ORDERING, topk_overlap = TOPK_OVERLAP)

X_aug_kipo, y_aug_kipo, info_kipo = kipo.select(X_train, y_train, X_test, y_test, ratio = BALANCE_RATIO, generator = "smotenc", 
                                                preprocessing = pipeline, **smotenc_config["params"])   

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

Generation of augmented data with K-IPO
Kendall's Tau threshold: 0.95, Topk ordering: 2, Topk overlap: 4
[INFO] total number of requested samples: 5515, generator: smotenc, importance mode: f-score_ANOVA, max block size: 2500, min block size: 50
[INFO] kendall's tau threshold: 0.95, topk overlap: 4, topk ordering: 2


To evaluate the explainability capabilities of the three models used in this study, it is necessary to specify how many input features are considered important for each dataset. In this work, this is defined as the minimum number of features that, collectively, account for at least 90% of the total ANOVA F-score. For the **AI4I2020** dataset, this (minimum) set includes **3** features.

In [23]:
MODELS = demo_config["models"]; XAI_METHODS = demo_config["XAI_methods"]
print(f"Evaluation will be conducted using the models: {MODELS}\n"
      f"Feature importance will be extracted using the XAI methods: {XAI_METHODS}"
)

num_topk_features = get_num_topk_features(DATASET)
print(f"\n[XAI INFO] Total number of important features for dataset {DATASET}: {num_topk_features}")

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

Evaluation will be conducted using the models: ['RandomForest', 'MLP', 'GAMINet']
Feature importance will be extracted using the XAI methods: ['shap', 'sofi', 'estimator_feature_importance']

[XAI INFO] Total number of important features for dataset ai4i2020: 3


In [24]:
X_train_aug_ctgan, X_test_aug_ctgan, y_train_aug_ctgan, y_test_aug_ctgan = train_test_split(X_aug_ctgan, y_aug_ctgan,train_size = TRAIN_TEST_RATIO, 
                                                                                            random_state = SEED_GEN, stratify = y_aug_ctgan)  

X_train_aug_tvae, X_test_aug_tvae, y_train_aug_tvae, y_test_aug_tvae = train_test_split(X_aug_tvae, y_aug_tvae,train_size = TRAIN_TEST_RATIO, 
                                                                                            random_state = SEED_GEN, stratify = y_aug_tvae) 

X_train_aug_gaussian_copula, X_test_aug_gaussian_copula, y_train_aug_gaussian_copula, y_test_aug_gaussian_copula = train_test_split(X_aug_gaussian_copula, y_aug_gaussian_copula,
                                                                                                                                    train_size = TRAIN_TEST_RATIO, random_state = SEED_GEN, 
                                                                                                                                    stratify = y_aug_gaussian_copula)

X_train_aug_smotenc, X_test_aug_smotenc, y_train_aug_smotenc, y_test_aug_smotenc = train_test_split(X_aug_smotenc, y_aug_smotenc, train_size = TRAIN_TEST_RATIO, 
                                                                                                    random_state = SEED_GEN, stratify = y_aug_smotenc)

X_train_aug_kipo, X_test_aug_kipo, y_train_aug_kipo, y_test_aug_kipo = train_test_split(X_aug_kipo, y_aug_kipo, train_size = TRAIN_TEST_RATIO, 
                                                                                        random_state = SEED_GEN, stratify = y_aug_kipo)

print(f"After splitting the augmented data with train/test ratio: {TRAIN_TEST_RATIO}")
print(f"CTGAN -> Training set samples: {X_train_aug_ctgan.shape[0]}, Test set samples: {X_test_aug_ctgan.shape[0]}")
print(f"TVAE -> Training set samples: {X_train_aug_tvae.shape[0]}, Test set samples: {X_test_aug_tvae.shape[0]}")
print(f"GaussianCopula -> Training set samples: {X_train_aug_gaussian_copula.shape[0]}, Test set samples: {X_test_aug_gaussian_copula.shape[0]}")
print(f"SMOTENC -> Training set samples: {X_train_aug_smotenc.shape[0]}, Test set samples: {X_test_aug_smotenc.shape[0]}")
print(f"K-IPO -> Training set samples: {X_train_aug_kipo.shape[0]}, Test set samples: {X_test_aug_kipo.shape[0]}")

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

After splitting the augmented data with train/test ratio: 0.8
CTGAN -> Training set samples: 10799, Test set samples: 2700
TVAE -> Training set samples: 10799, Test set samples: 2700
GaussianCopula -> Training set samples: 10799, Test set samples: 2700
SMOTENC -> Training set samples: 10799, Test set samples: 2700
K-IPO -> Training set samples: 10799, Test set samples: 2700


After the data generated by each generator is split into train and test sets, all three estimators are trained and evaluated on each such pair. Various metrics related to predictive capabilities are recorded, including accuracy, F1 score and PR AUC. 

Additionally, for each estimator, the corresponding XAI technique is applied to extract the feature importance ranking for the input features, which is then used to evaluate the estimator’s explainability capabilities. The mapping of estimators to XAI techniques is as follows:

- Random Forest: SHAP

- MLP: SOFI

- GAMI-Net: Direct extraction from the model (inherently interpretable)

To evaluate explainability, a ground truth is required. In this work, the ground truth is defined as the feature importance ranking derived from **seven model-independent techniques**:

- Distance Correlation

- Mutual Information (MI)

- Laplacian Score

- ReliefF

- Gini Index

- Joint Mutual Information (JMI)

- Maximum Relevance Maximum Independence (MRI) 

For each of these seven techniques, the overlap percentage with the estimator’s importance ranking (for the **important features** of the dataset) is calculated. The explainability capability of each estimator is then defined as the average of these overlap percentages.

For each generator’s dataset, once model training and evaluation are complete, the following are computed:

- The average predictive metrics across the three estimators are calculated (**final predictive capability**)

- The average feature importance overlap percentages for the estimators are calculated (**final explainability capability**)

In [25]:
pred_results_ctgan = []; xai_results_ctgan = []

for i, estimator_name in enumerate(MODELS):
    model_conf = models[estimator_name]
    estimator = clone(model_conf["model"](**model_conf["params"]))
    estimator.fit(X_train_aug_ctgan, y_train_aug_ctgan.values.ravel())

    probs = estimator.predict_proba(X_test_aug_ctgan)[:, -1]
    preds = probs >= calculate_threshold(y_test_aug_ctgan, probs, criterion = "geo_mean")

    pred_metrics = compute_metrics(probs, preds, y_test_aug_ctgan)
    xai_metrics = compute_explanations(X_train = X_train_aug_ctgan, y_train = y_train_aug_ctgan.values.ravel(), estimator = estimator,
                                       X_test = X_test_aug_ctgan, y_test = y_test_aug_ctgan.values.ravel(), num_cols = num_cols, 
                                       cat_cols = cat_cols, mode = XAI_METHODS[i])

    pred_results_ctgan.append(pred_metrics); xai_results_ctgan.append(xai_metrics)

final_pred_results_ctgan = compute_final_results(pred_results_ctgan)
final_importance_ctgan = format_feature_importance(xai_results_ctgan, XAI_METHODS)
importance_overlap_perc_ctgan = compute_importance_overlap(final_importance_ctgan, DATASET, topk = num_topk_features) 

separability_ctgan = separability_N3(X_train_aug_ctgan, y_train_aug_ctgan.values.ravel()) 

print(f"[INFO -> CTGAN] Training and testing of all models completed successfully")

[INFO -> CTGAN] Training and testing of all models completed successfully


In [26]:
pred_results_tvae = []; xai_results_tvae = []

for i, estimator_name in enumerate(MODELS):
    model_conf = models[estimator_name]
    estimator = clone(model_conf["model"](**model_conf["params"]))
    estimator.fit(X_train_aug_tvae, y_train_aug_tvae.values.ravel())

    probs = estimator.predict_proba(X_test_aug_tvae)[:, -1]
    preds = probs >= calculate_threshold(y_test_aug_tvae, probs, criterion = "geo_mean")

    pred_metrics = compute_metrics(probs, preds, y_test_aug_tvae)
    xai_metrics = compute_explanations(X_train = X_train_aug_tvae, y_train = y_train_aug_tvae.values.ravel(), estimator = estimator,
                                       X_test = X_test_aug_tvae, y_test = y_test_aug_tvae.values.ravel(), num_cols = num_cols, 
                                       cat_cols = cat_cols, mode = XAI_METHODS[i])

    pred_results_tvae.append(pred_metrics); xai_results_tvae.append(xai_metrics)

final_pred_results_tvae = compute_final_results(pred_results_tvae)
final_importance_tvae = format_feature_importance(xai_results_tvae, XAI_METHODS)
importance_overlap_perc_tvae = compute_importance_overlap(final_importance_tvae, DATASET, topk = num_topk_features)

separability_tvae = separability_N3(X_train_aug_tvae, y_train_aug_tvae.values.ravel()) 

print(f"[INFO -> TVAE] Training and testing of all models completed successfully")

[INFO -> TVAE] Training and testing of all models completed successfully


In [27]:
pred_results_gaussian_copula = []; xai_results_gaussian_copula = []

for i, estimator_name in enumerate(MODELS):
    model_conf = models[estimator_name]
    estimator = clone(model_conf["model"](**model_conf["params"]))
    estimator.fit(X_train_aug_gaussian_copula, y_train_aug_gaussian_copula.values.ravel())

    probs = estimator.predict_proba(X_test_aug_gaussian_copula)[:, -1]
    preds = probs >= calculate_threshold(y_test_aug_gaussian_copula, probs, criterion = "geo_mean")

    pred_metrics = compute_metrics(probs, preds, y_test_aug_gaussian_copula)
    xai_metrics = compute_explanations(X_train = X_train_aug_gaussian_copula, y_train = y_train_aug_gaussian_copula.values.ravel(), estimator = estimator,
                                       X_test = X_test_aug_gaussian_copula, y_test = y_test_aug_gaussian_copula.values.ravel(), num_cols = num_cols, 
                                       cat_cols = cat_cols, mode = XAI_METHODS[i])

    pred_results_gaussian_copula.append(pred_metrics); xai_results_gaussian_copula.append(xai_metrics)

final_pred_results_gaussian_copula = compute_final_results(pred_results_gaussian_copula)
final_importance_gaussian_copula = format_feature_importance(xai_results_gaussian_copula, XAI_METHODS)
importance_overlap_perc_gaussian_copula = compute_importance_overlap(final_importance_gaussian_copula, DATASET, topk = num_topk_features)

separability_gaussian_copula = separability_N3(X_train_aug_gaussian_copula, y_train_aug_gaussian_copula.values.ravel()) 

print(f"[INFO -> Gaussian Copula] Training and testing of all models completed successfully")

[INFO -> Gaussian Copula] Training and testing of all models completed successfully


In [28]:
pred_results_smotenc = []; xai_results_smotenc = []

for i, estimator_name in enumerate(MODELS):
    model_conf = models[estimator_name]
    estimator = clone(model_conf["model"](**model_conf["params"]))
    estimator.fit(X_train_aug_smotenc, y_train_aug_smotenc.values.ravel())

    probs = estimator.predict_proba(X_test_aug_smotenc)[:, -1]
    preds = probs >= calculate_threshold(y_test_aug_smotenc, probs, criterion = "geo_mean")

    pred_metrics = compute_metrics(probs, preds, y_test_aug_smotenc)
    xai_metrics = compute_explanations(X_train = X_train_aug_smotenc, y_train = y_train_aug_smotenc.values.ravel(), estimator = estimator,
                                       X_test = X_test_aug_smotenc, y_test = y_test_aug_smotenc.values.ravel(), num_cols = num_cols, 
                                       cat_cols = cat_cols, mode = XAI_METHODS[i])

    pred_results_smotenc.append(pred_metrics); xai_results_smotenc.append(xai_metrics)

final_pred_results_smotenc = compute_final_results(pred_results_smotenc)
final_importance_smotenc = format_feature_importance(xai_results_smotenc, XAI_METHODS)
importance_overlap_perc_smotenc = compute_importance_overlap(final_importance_smotenc, DATASET, topk = num_topk_features)

separability_smotenc = separability_N3(X_train_aug_smotenc, y_train_aug_smotenc.values.ravel()) 

print(f"[INFO -> SMOTENC] Training and testing of all models completed successfully")

[INFO -> SMOTENC] Training and testing of all models completed successfully


In [29]:
pred_results_kipo = []; xai_results_kipo = []

for i, estimator_name in enumerate(MODELS):
    model_conf = models[estimator_name]
    estimator = clone(model_conf["model"](**model_conf["params"]))
    estimator.fit(X_train_aug_kipo, y_train_aug_kipo.values.ravel())

    probs = estimator.predict_proba(X_test_aug_kipo)[:, -1]
    preds = probs >= calculate_threshold(y_test_aug_kipo, probs, criterion = "geo_mean")

    pred_metrics = compute_metrics(probs, preds, y_test_aug_kipo)
    xai_metrics = compute_explanations(X_train = X_train_aug_kipo, y_train = y_train_aug_kipo.values.ravel(), estimator = estimator,
                                       X_test = X_test_aug_kipo, y_test = y_test_aug_kipo.values.ravel(), num_cols = num_cols, 
                                       cat_cols = cat_cols, mode = XAI_METHODS[i])

    pred_results_kipo.append(pred_metrics); xai_results_kipo.append(xai_metrics)

final_pred_results_kipo = compute_final_results(pred_results_kipo)
final_importance_kipo = format_feature_importance(xai_results_kipo, XAI_METHODS)
importance_overlap_perc_kipo = compute_importance_overlap(final_importance_kipo, DATASET, topk = num_topk_features)

separability_kipo = separability_N3(X_train_aug_kipo, y_train_aug_kipo.values.ravel()) 

print(f"[INFO -> K-IPO] Training and testing of all models completed successfully")

[INFO -> K-IPO] Training and testing of all models completed successfully


In [30]:
final_importances = {
    "CTGAN": final_importance_ctgan,
    "TVAE": final_importance_tvae,
    "Gaussian Copula": final_importance_gaussian_copula,
    "SMOTENC": final_importance_smotenc,
    "K-IPO": final_importance_kipo
}

print("Feature Importance Rankings after fitting (RandomForest -> SHAP, MLP -> SOFI, GAMI-Net -> Estimator Feature Importance):")

for generator_name, final_importance in final_importances.items():
    print(f"\n-> {generator_name}")
    
    cols_cl = []
    for col in final_importance.columns:
        if '-' in col or '_' in col:
            col = col.replace('-', ' ').replace('_', ' ').title()
        else:
            col = col.upper()
        cols_cl.append(col)
        

    df_ranking = pd.DataFrame({new_col: final_importance[orig_col].tolist()
                               for new_col, orig_col in zip(cols_cl, final_importance.columns)})
    df_ranking.index = range(1, len(df_ranking) + 1)
    
    display(
        df_ranking.style.set_properties(**{'text-align': 'center'})
                        .set_table_styles([dict(selector='th', props=[('text-align', 'center')])])
    )

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

Feature Importance Rankings after fitting (RandomForest -> SHAP, MLP -> SOFI, GAMI-Net -> Estimator Feature Importance):

-> CTGAN


,SHAP,SOFI,Estimator Feature Importance
1,Rotational speed [rpm],Torque [Nm],Rotational speed [rpm]
2,Torque [Nm],Tool wear [min],Type
3,Type,Air temperature [K],Tool wear [min]
4,Tool wear [min],Process temperature [K],Torque [Nm]
5,Air temperature [K],Rotational speed [rpm],Air temperature [K]
6,Process temperature [K],Type,Process temperature [K]



-> TVAE


,SHAP,SOFI,Estimator Feature Importance
1,Rotational speed [rpm],Torque [Nm],Rotational speed [rpm]
2,Torque [Nm],Tool wear [min],Type
3,Type,Air temperature [K],Tool wear [min]
4,Tool wear [min],Process temperature [K],Torque [Nm]
5,Air temperature [K],Rotational speed [rpm],Air temperature [K]
6,Process temperature [K],Type,Process temperature [K]


...

,SHAP,SOFI,Estimator Feature Importance
1,Rotational speed [rpm],Torque [Nm],Rotational speed [rpm]
2,Torque [Nm],Tool wear [min],Torque [Nm]
3,Tool wear [min],Air temperature [K],Tool wear [min]
4,Air temperature [K],Process temperature [K],Air temperature [K]
5,Process temperature [K],Rotational speed [rpm],Process temperature [K]
6,Type,Type,Type



-> SMOTENC


,SHAP,SOFI,Estimator Feature Importance
1,Torque [Nm],Tool wear [min],Torque [Nm]
2,Rotational speed [rpm],Type,Tool wear [min]
3,Tool wear [min],Air temperature [K],Rotational speed [rpm]
4,Air temperature [K],Process temperature [K],Air temperature [K]
5,Type,Rotational speed [rpm],Process temperature [K]
6,Process temperature [K],Torque [Nm],Type



-> K-IPO


,SHAP,SOFI,Estimator Feature Importance
1,Rotational speed [rpm],Tool wear [min],Torque [Nm]
2,Torque [Nm],Rotational speed [rpm],Rotational speed [rpm]
3,Tool wear [min],Torque [Nm],Tool wear [min]
4,Air temperature [K],Type,Air temperature [K]
5,Process temperature [K],Air temperature [K],Process temperature [K]
6,Type,Process temperature [K],Type


For each augmented dataset, the Kendall tau coefficient and the number of top-k features with unchanged importance ranking are calculated.

In [31]:
aug_importance_ctgan = compute_importance(mode = "f-score_ANOVA", X_train = X_aug_ctgan, y_train = y_aug_ctgan.values.ravel())  
kendall_tau_ctgan = kendall_tau(raw_importance, aug_importance_ctgan)
topk_ordering_ctgan = compute_topk_ordering(raw_importance, aug_importance_ctgan)

aug_importance_tvae = compute_importance(mode = "f-score_ANOVA", X_train = X_aug_tvae, y_train = y_aug_tvae.values.ravel())  
kendall_tau_tvae = kendall_tau(raw_importance, aug_importance_tvae)
topk_ordering_tvae = compute_topk_ordering(raw_importance, aug_importance_tvae)

aug_importance_gaussian_copula = compute_importance(mode = "f-score_ANOVA", X_train = X_aug_gaussian_copula, y_train = y_aug_gaussian_copula.values.ravel())  
kendall_tau_gaussian_copula = kendall_tau(raw_importance, aug_importance_gaussian_copula)
topk_ordering_gaussian_copula = compute_topk_ordering(raw_importance, aug_importance_gaussian_copula)

aug_importance_smotenc = compute_importance(mode = "f-score_ANOVA", X_train = X_aug_smotenc, y_train = y_aug_smotenc.values.ravel())  
kendall_tau_smotenc = kendall_tau(raw_importance, aug_importance_smotenc)
topk_ordering_smotenc = compute_topk_ordering(raw_importance, aug_importance_smotenc)

aug_importance_kipo = compute_importance(mode = "f-score_ANOVA", X_train = X_aug_kipo, y_train = y_aug_kipo.values.ravel())  
kendall_tau_kipo = kendall_tau(raw_importance, aug_importance_kipo)
topk_ordering_kipo = compute_topk_ordering(raw_importance, aug_importance_kipo)

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

In [32]:
ctgan_metrics_dict = {
    "Kendall's Tau": kendall_tau_ctgan,
    "Top-k ordering": topk_ordering_ctgan,
    **final_pred_results_ctgan,
    "Separability": separability_ctgan,
    "Top-k overlap (%)": importance_overlap_perc_ctgan
}

tvae_metrics_dict = {
    "Kendall's Tau": kendall_tau_tvae,
    "Top-k ordering": topk_ordering_tvae,
    **final_pred_results_tvae,
    "Separability": separability_tvae,
    "Top-k overlap (%)": importance_overlap_perc_tvae
}

gc_metrics_dict = {
    "Kendall's Tau": kendall_tau_gaussian_copula,
    "Top-k ordering": topk_ordering_gaussian_copula,
    **final_pred_results_gaussian_copula,
    "Separability": separability_gaussian_copula,
    "Top-k overlap (%)": importance_overlap_perc_gaussian_copula
}

smotenc_metrics_dict = {
    "Kendall's Tau": kendall_tau_smotenc,
    "Top-k ordering": topk_ordering_smotenc,
    **final_pred_results_smotenc,
    "Separability": separability_smotenc,
    "Top-k overlap (%)": importance_overlap_perc_smotenc
}

kipo_metrics_dict = {
    "Kendall's Tau": kendall_tau_kipo,
    "Top-k ordering": topk_ordering_kipo,
    **final_pred_results_kipo,
    "Separability": separability_kipo,
    "Top-k overlap (%)": importance_overlap_perc_kipo
}

print("Final Evaluation Results:")

results_combined = pd.DataFrame(
    [ctgan_metrics_dict, tvae_metrics_dict, gc_metrics_dict, smotenc_metrics_dict, kipo_metrics_dict],
    index = ["CTGAN", "TVAE", "Gaussian Copula", "SMOTENC", "K-IPO"]
)


format_dict = {col : "{:d}" if col == "Top-k ordering" else "{:.3f}" for col in results_combined.columns}

display(
    results_combined.style.format(format_dict)
                    .set_properties(**{'text-align' : 'center'})
                    .set_table_styles([dict(selector = 'th', props = [('text-align', 'center')])])
)

HTML(value='\n        <style>\n\n        .left-label {\n            width: 30%;\n        }\n\n        .card-pa…

Final Evaluation Results:


,Kendall's Tau,Top-k ordering,Accuracy,Precision,F1,Recall,ROC_AUC,PR_AUC,MCC,BalancedAcc,Separability,Top-k overlap (%)
CTGAN,0.467,1,0.944,0.914,0.936,0.960,0.987,0.981,0.887,0.946,0.903,63.492
TVAE,0.467,1,0.945,0.906,0.938,0.972,0.987,0.982,0.890,0.948,0.913,63.492
Gaussian Copula,0.733,1,0.968,0.968,0.962,0.955,0.991,0.991,0.934,0.966,0.943,85.714
SMOTENC,0.733,2,0.972,0.958,0.968,0.978,0.995,0.993,0.943,0.973,0.977,74.603
K-IPO,1.000,6,0.979,0.971,0.976,0.982,0.996,0.994,0.958,0.980,0.982,90.476
